# 찍먹 | 딥러닝 코드 분석 — 최종 모델 정리
- 음식 사진을 보고 알레르기 재료를 알려 주는 앱 **찍먹**에서 사용하는 딥러닝 모델 두 개를 정리함.
- **코드는 최종으로 채택한 모델만** 실행하고, 그 결론에 이르기까지의 실험은 5장에 글과 표로 정리함.

### 최종 구성 한눈에 보기
| 역할 | 최종 모델 | 학습 방식 | 핵심 결과 |
|---|---|---|---|
| 아는 음식 101종 분류 | **ConvNeXt-Base (exp004)** | ImageNet 사전학습 가중치 → Food-101 **전체 파인튜닝** | 시험 Top-1 **93.46%** |
| 처음 보는 음식의 레시피 검색 | **기본 CLIP ViT-B/32** | 추가 학습 없음 (OpenAI 공개 사전학습 모델) | 처음 보는 음식 주의놓침 **0.090** (비교한 모델 중 가장 낮음) |

### 실행 방법
Colab에서 GPU 런타임으로 `모두 실행`. 앱이 쓰는 최종 가중치와 검색 목록은 아래 폴더에서 불러오고, Food-101 데이터는 용량(약 5GB) 때문에 폴더에 넣지 않고 실행할 때 공식 주소에서 자동으로 받는다.

```
[딥러닝]최종제출_5조/
├── 찍먹_딥러닝_코드정리.ipynb   ← 이 노트북
├── weights/best.pt               ConvNeXt-Base exp004 최종 가중치 (앱과 동일)
└── clip_base/                    CLIP 레시피 검색 목록 (앱과 동일)
    ├── index_emb.npy
    └── index_meta.parquet
```

In [ ]:
%pip -q install timm

## 1. 데이터 — Food-101
**Food-101** (Bossard et al., 2014): 음식 101종 × 1,000장, 약 10만 장의 공개 데이터셋. 공식적으로 음식당 750장은 학습용, 250장은 시험용으로 나뉘어 있다.

| 구분 | 장수 | 만든 방법 |
|---|---:|---|
| 학습 | 68,172 | 공식 train에서 음식당 75장을 뺀 나머지 |
| 검증(모의고사) | 7,575 | 공식 train에서 음식당 75장 → 설정·에폭 선택에 사용 |
| 시험 | 25,250 | 공식 test → **최종 평가에만 한 번 사용** |

- 라벨이 잘못된 사진 3장은 제외했다.
- 시험 데이터로 설정을 고르면 시험 점수가 실제보다 좋게 나오므로, 설정 비교는 검증 데이터로만 했다.
- 데이터는 용량(약 5GB) 때문에 제출 폴더에 넣지 않았다. 실행하면 Food-101 **공식 배포 주소에서 Colab 로컬 디스크로 자동 다운로드**하고 압축을 푼다(처음 한 번, 약 5~10분). 학습 때와 같은 원본 파일이다.

In [ ]:
import random, shutil, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import torch, timm
from torch import nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from timm.data import resolve_data_config, create_transform

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('실행 장치:', DEVICE)
if DEVICE.type != 'cuda':
    print('주의: CPU에서는 시험 데이터 2만 5천 장 평가가 오래 걸립니다. GPU 런타임을 권장합니다.')

# ── 제출 폴더 찾기 (가중치·CLIP 검색 목록용) ──
SUBMIT_NAME = '[딥러닝]최종제출_5조'
DATA_ROOT = Path('./data_local')          # Food-101을 받을 Colab 로컬 위치

def same_name(a, b):
    # Drive는 한글을 자모 분리(NFD)로 줄 때가 있어 NFC로 맞춰 비교
    return unicodedata.normalize('NFC', a) == unicodedata.normalize('NFC', b)

def find_submit_folder():
    if (Path('weights')/'best.pt').is_file():
        return Path('.').resolve()        # 로컬 Jupyter에서 노트북 폴더로 실행한 경우
    if Path('/content').is_dir():         # Colab: Drive 연결 후 폴더 찾기
        if not Path('/content/drive').is_dir():
            from google.colab import drive
            drive.mount('/content/drive')
        D = Path('/content/drive')
        # 폴더 이름의 [ ]가 glob 특수문자라서, 목록을 받은 뒤 이름을 직접 비교
        for pat in ['MyDrive/*', 'MyDrive/*/*', 'Shareddrives/*/*',
                    'MyDrive/.shortcut-targets-by-id/*/*', 'MyDrive/.shortcut-targets-by-id/*/*/*']:
            for p in D.glob(pat):
                if same_name(p.name, SUBMIT_NAME) and p.is_dir():
                    return p
    raise FileNotFoundError(f"'{SUBMIT_NAME}' 폴더를 찾지 못했습니다. 공유받은 폴더라면 "
                            "'드라이브에 바로가기 추가' 후 다시 실행해 주세요.")

def find_food101(root):
    if not Path(root).is_dir(): return None
    for classes_txt in sorted(Path(root).glob('**/meta/classes.txt')):
        if (classes_txt.parent.parent/'images').is_dir():
            return classes_txt.parent.parent
    return None

def prepare_food101():
    found = find_food101(DATA_ROOT)
    if found: return found                # 이미 받아 둔 경우
    # 데이터는 용량(약 5GB) 때문에 제출 폴더에 넣지 않고, 공식 배포 주소에서 받는다 (학습 때와 같은 원본)
    from torchvision.datasets import Food101
    print('Food-101 다운로드 중 (처음 한 번, 약 5~10분)')
    Food101(root=str(DATA_ROOT), split='test', download=True)   # 다운로드·파일 검사·압축 해제 자동
    return find_food101(DATA_ROOT)

ROOT = find_submit_folder()
FOOD = prepare_food101()
print('제출 폴더:', ROOT)

# ── 학습 / 검증 / 시험 나누기 (학습 때와 같은 규칙) ──
CLASSES = (FOOD/'meta/classes.txt').read_text().split()
CIDX = {name: i for i, name in enumerate(CLASSES)}
EXCLUDE = {'steak/1340977', 'lasagna/3787908', 'bread_pudding/1375816'}   # 라벨 오류 3장
train_ids = [s for s in (FOOD/'meta/train.txt').read_text().split() if s not in EXCLUDE]
test_ids = (FOOD/'meta/test.txt').read_text().split()
by_class = {c: sorted(s for s in train_ids if s.split('/')[0] == c) for c in CLASSES}
rng = random.Random(42)
for ids in by_class.values(): rng.shuffle(ids)
val_ids = [s for c in CLASSES for s in by_class[c][:75]]
tr_ids = [s for c in CLASSES for s in by_class[c][75:]]
print('학습/검증/시험:', len(tr_ids), len(val_ids), len(test_ids))

## 2. 최종 분류 모델 — ConvNeXt-Base (exp004)

### 왜 사전학습 모델을 썼나? (전이학습)
음식 사진을 처음부터 학습시키려면 아주 많은 데이터와 연산이 필요하다. 대규모 이미지(ImageNet)로 이미 모양·색·질감 같은 특징을 배운 가중치를 가져와 Food-101에 맞게 다시 학습시키면 훨씬 효율적이다. 이것이 **전이학습**이다.
- 사용한 공개 가중치: `convnext_base.fb_in22k_ft_in1k` (ImageNet-22K로 학습 → ImageNet-1K로 미세조정된 **공개 체크포인트**. 이 단계는 팀이 한 것이 아니다.)
- 팀이 한 일: 마지막 출력층을 ImageNet 1,000종 → **음식 101종**으로 바꾸고, **모든 층을 Food-101로 파인튜닝**했다.

### 왜 ConvNeXt-Base인가?
가벼운 Tiny로 먼저 기준선(시험 89.19%)을 만들고, 정확도를 더 올리기 위해 Base로 키운 뒤 학습 설정을 바꿔 가며 비교했다(5장). 검증 Top-1이 가장 높았던 **exp004**를 최종으로 정했고, 앱도 이 가중치를 쓴다.

### exp004 최종 학습 설정
| 항목 | 값 | 의미 |
|---|---|---|
| 학습 범위 | 전체 층 파인튜닝, layer decay 0.9 | 앞쪽 층(일반적 특징)은 더 작은 학습률로 조금만 바꿈 |
| 옵티마이저 | AdamW, lr 2e-4, weight decay 0.05 | |
| 배치 | 32 × 4 (gradient accumulation) = 128 | GPU 메모리 한계 안에서 큰 배치 효과 |
| 학습률 일정 | 1에폭 워밍업 후 cosine 감소 | |
| 데이터 증강 | RandAugment, 랜덤 크롭(면적 25~100%), 좌우 반전 | Mixup/CutMix는 **사용 안 함** |
| 정규화 | label smoothing 0.1, drop path 0.1 | 과적합 완화 |
| EMA | 0.999 | 가중치의 이동평균을 최종 모델로 사용 |
| 에폭 | 최대 24, 검증 Top-1이 4에폭 연속 0.05%p 이상 안 오르면 중단 | **12에폭에서 최고, 16에폭에서 중단** |
| 입력 크기 | 224 × 224 | 평가 시 중앙 크롭(crop_pct 0.875) |

아래 코드는 학습하지 않고, 이 설정으로 학습해 둔 **`weights/best.pt`를 바로 적용**한다.

In [ ]:
# 모델 구조: ConvNeXt-Base + 101종 출력층. 가중치는 학습해 둔 best.pt로 채우므로 사전학습 가중치는 받지 않음.
MODEL_NAME = 'convnext_base.fb_in22k_ft_in1k'
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=101)

ckpt_path = ROOT/'weights'/'best.pt'
checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
state = checkpoint
if isinstance(checkpoint, dict):
    for key in ('model', 'state_dict_ema', 'model_ema', 'state_dict'):
        if key in checkpoint and isinstance(checkpoint[key], dict):
            state = checkpoint[key]; break
state = {k.removeprefix('module.'): v for k, v in state.items()}
model.load_state_dict(state)          # 구조가 다르면 여기서 에러 → 다른 가중치가 섞이지 않음
model = model.to(DEVICE).eval()
print('최종 가중치 적용:', ckpt_path)
if isinstance(checkpoint, dict):
    print('저장 당시 에폭:', checkpoint.get('epoch', '기록 없음'),
          '| 검증 Top-1:', checkpoint.get('val_top1', '기록 없음'))

# 평가용 전처리: 학습 때와 같은 정규화, 224 중앙 크롭
config = resolve_data_config({}, model=model)
eval_tf = create_transform(224, is_training=False, crop_pct=0.875,
                           interpolation=config['interpolation'],
                           mean=config['mean'], std=config['std'])

class FoodImages(Dataset):
    def __init__(self, ids, transform): self.ids, self.transform = ids, transform
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        path = self.ids[i]
        with Image.open(FOOD/'images'/f'{path}.jpg') as img:
            x = self.transform(img.convert('RGB'))
        return x, CIDX[path.split('/')[0]]

# T4 등은 bf16 합성곱을 지원하지 않아 cuDNN 에러가 남 → Ampere(8.x) 이상에서만 bf16, 나머지는 fp16
use_bf16 = DEVICE.type == 'cuda' and torch.cuda.get_device_capability()[0] >= 8
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

@torch.no_grad()
def evaluate(net, loader):
    net.eval(); n = top1 = top5 = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=amp_dtype, enabled=DEVICE.type == 'cuda'):
            logits = net(x)
        pred = logits.float().topk(5, dim=1).indices
        top1 += (pred[:, 0] == y).sum().item()
        top5 += (pred == y[:, None]).any(dim=1).sum().item()
        n += len(y)
    return {'top1': 100 * top1 / n, 'top5': 100 * top5 / n}

### 시험 데이터 평가
최종 모델로 **시험 데이터 25,250장을 한 번** 평가한다. 원본 기록은 시험 Top-1 **93.46%** 이며, 같은 가중치·같은 전처리이므로 아래 결과도 이와 같거나 매우 가까워야 한다.

In [ ]:
test_loader = DataLoader(FoodImages(test_ids, eval_tf), batch_size=64, num_workers=2,
                         pin_memory=DEVICE.type == 'cuda')
result = evaluate(model, test_loader)
print(f"시험 Top-1: {result['top1']:.2f}%   Top-5: {result['top5']:.2f}%   (원본 기록 Top-1 93.46%)")

### 사진 한 장 예측해 보기
앱에서처럼 사진 한 장을 넣으면 확률이 높은 음식 3개를 보여 준다. (여기 확률은 보정 전 softmax 값이다. 보정은 4장 참고)

In [ ]:
@torch.no_grad()
def predict(image, k=3):
    x = eval_tf(image.convert('RGB')).unsqueeze(0).to(DEVICE)
    prob = model(x).float().softmax(dim=1)[0]
    p, idx = prob.topk(k)
    return pd.DataFrame({'음식': [CLASSES[i] for i in idx.tolist()], '확률': [round(v, 4) for v in p.tolist()]})

sample_id = test_ids[1]                  # 다른 사진을 보려면 번호를 바꾸면 된다
sample = Image.open(FOOD/'images'/f'{sample_id}.jpg')
display(sample.resize((224, 224)))
print('실제 음식:', sample_id.split('/')[0])
display(predict(sample))

## 3. 최종 검색 모델 — 기본 CLIP ViT-B/32

### 왜 CLIP이 필요한가?
ConvNeXt는 아는 101종 중 하나를 **반드시** 골라야 한다. 그래서 처음 보는 음식(예: 카오소이)을 비슷한 음식(치킨 커리)으로 잘못 답하고, 그 음식의 알레르기 재료를 놓칠 수 있다.
CLIP은 대규모 **이미지–텍스트 쌍**으로 미리 학습된 모델이라, 사진과 글을 같은 공간의 숫자(임베딩)로 바꿔 비교할 수 있다. 그래서 101종 밖의 음식도 **사진과 가장 비슷한 레시피 글**을 찾아 재료를 확인할 수 있다.

### 왜 추가 학습하지 않은 기본 CLIP인가?
레시피 사진으로 CLIP을 세 번 추가 학습해 봤지만, 가장 중요한 기준인 **처음 보는 음식의 알레르기 주의놓침**이 모두 기본 CLIP보다 늘었다(5장). 그래서 공개 모델 `openai/clip-vit-base-patch32`를 그대로 쓴다. ViT-B/32는 앱의 CPU 추론 속도까지 고려한 크기다.

### 검색 대상
레시피 100만 개(`1M_Final.csv`)에서 우리 음식 **175종**의 레시피를 골라, 음식당 최대 300개씩 **35,184개 레시피 글(제목 + 재료)** 을 검색 목록으로 만들었다. 레시피 글은 미리 CLIP 임베딩으로 바꿔 `clip_base/`에 저장해 두었고(앱과 동일), 사진이 들어오면 사진만 임베딩으로 바꿔 **코사인 유사도가 높은 30개**를 찾는다.

In [ ]:
from transformers import CLIPModel
from torchvision import transforms as T

CLIP_NAME = 'openai/clip-vit-base-patch32'          # 공개 사전학습 모델 (추가 학습 없음)
clip = CLIPModel.from_pretrained(CLIP_NAME).to(DEVICE).eval()
clip_tf = T.Compose([T.Resize(224, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224),
                     T.ToTensor(),
                     T.Normalize([0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711])])

@torch.inference_mode()
def image_embedding(image):
    pixels = clip_tf(image.convert('RGB')).unsqueeze(0).to(DEVICE)
    pooled = clip.vision_model(pixel_values=pixels).pooler_output
    return F.normalize(clip.visual_projection(pooled).float(), dim=-1).cpu()

def search_recipes(image, k=30):
    similarity = (image_embedding(image) @ text_vectors.T).squeeze(0)
    scores, ids = similarity.topk(min(k, len(index_meta)))
    result = index_meta.iloc[ids.tolist()].copy()
    result.insert(0, 'similarity', [round(v, 4) for v in scores.tolist()])
    return result

# 앱과 같은 검색 목록 불러오기
emb_path, meta_path = ROOT/'clip_base'/'index_emb.npy', ROOT/'clip_base'/'index_meta.parquet'
if emb_path.is_file() and meta_path.is_file():
    text_vectors = F.normalize(torch.from_numpy(np.load(emb_path)).float(), dim=-1)
    index_meta = pd.read_parquet(meta_path).reset_index(drop=True)
    assert len(index_meta) == len(text_vectors), '임베딩 수와 레시피 정보 수가 다릅니다.'
    print('검색 목록:', len(index_meta), '개 레시피 | 열:', list(index_meta.columns))
    display(sample.resize((224, 224)))
    print('실제 음식:', sample_id.split('/')[0], '→ 비슷한 레시피 상위 10개')
    display(search_recipes(sample).head(10))
else:
    print('clip_base/ 검색 목록이 없어 CLIP 검색 실행을 건너뜁니다.')

## 4. 앱에서 두 모델을 함께 쓰는 방법
1. 사진이 들어오면 ConvNeXt가 101종 중 확률을 계산한다.
2. 확률을 **Temperature Scaling**으로 보정한다. (원래 모델이 자기 확률을 실제보다 높거나 낮게 말하는 것을 바로잡는 방법)
3. 보정 확률에 따라:
   - **0.95 이상:** 1등 음식으로 판단
   - **0.6 ~ 0.95:** 상위 3개 음식을 함께 고려
   - **0.6 미만:** 처음 보는 음식일 수 있으므로 **CLIP 레시피 검색**으로 연결
4. 확률이 높아도 CLIP으로 한 번 더 확인하도록 설계했다.

**확률만으로는 부족했던 이유 (원본 기록):** 처음 보는 음식을 확률로 걸러내는 OOD 탐지(MSP/Energy)는 AUROC 약 0.86\~0.88, FPR@TPR95 약 0.41\~0.49로, 처음 보는 음식을 확률만으로 안전하게 걸러내기 어려웠다. 그래서 CLIP 검색을 함께 쓴다.

> Temperature Scaling 값(T=0.838, ECE 0.0651 → 0.0174)은 **exp003 모델**에서 구한 기록이다. 최종 exp004에 쓰려면 검증 데이터로 다시 구해야 하며, 이 노트북에서는 다시 계산하지 않았다.

## 5. 최종 결정까지의 실험 기록 (텍스트 정리)
아래 수치는 모두 **원본 프로젝트의 실험 기록**이며, 이 노트북에서 다시 실행한 결과가 아니다.

### 5-1. ConvNeXt — 무엇을 바꿨고 왜 exp004인가?
| 실험 | 이전 실험에서 바꾼 것 | 검증 Top-1 | 시험 Top-1 |
|---|---|---:|---:|
| Tiny 기준선 | ImageNet-1K 사전학습 Tiny | 85.39% | 89.19% |
| exp001 | Base로 변경, Mixup 0.1 / CutMix 1.0, EMA 등 | 89.39% | 93.08% |
| exp002 | Mixup/CutMix 제거 | 89.40% | 93.15% |
| exp003 | 학습률 1e-4 → 2e-4 | 89.91% | 93.37% |
| **exp004 (최종)** | 최대 에폭 12 → 24, patience 4 | **90.36%** | **93.46%** |

- **Tiny → Base:** 시험 정확도가 약 4%p 올랐다. 다만 모델 크기와 함께 사전학습 데이터(ImageNet-22K 경유)와 학습 설정도 달라졌으므로, 개선 원인을 모델 크기만으로 단정할 수 없다.
- **Mixup/CutMix 제거:** 차이가 0.01%p로 매우 작아 반복 실험 없이 효과를 단정하기 어렵다. 이후 실험은 Mixup/CutMix 없이 진행했다.
- **학습률 2배:** 검증 Top-1이 0.51%p 올라 가장 뚜렷한 개선이었다.
- **에폭 늘리기:** 12에폭에서 최고를 찍고 16에폭에서 조기 종료됐다. 검증 기준 가장 좋아 최종으로 선택했다.
- 최종 선택은 **검증 점수**로 했고, 시험 점수는 확인용으로만 봤다.

### 5-2. CLIP — 추가 학습을 했지만 기본 모델을 고른 이유
**평가 기준:** 처음 보는 음식 사진으로 레시피를 찾았을 때 ① 정답 음식이 상위 5개 안에 드는 비율(Top-5), ② 19종 알레르기 중 주의가 필요한 재료를 놓친 비율(**주의놓침**). 앱에서는 알레르기를 놓치는 것이 가장 위험하므로 **주의놓침을 가장 중요하게** 봤다.

**추가 학습 데이터:** 레시피에 달린 사진 주소에서 모은 레시피 사진과, 그 레시피의 제목·재료 글을 짝지은 데이터.

| 실험 | 무엇을 바꿨나? | 처음 보는 음식 Top-5 | 주의놓침 |
|---|---|---:|---:|
| **기본 CLIP (최종)** | 추가 학습 없음 | 0.735 | **0.090** |
| 1차 (04) | 사진–레시피 글 대조학습, 전체 층 학습 | 0.683 | 0.115 |
| 2차 (04-2) | 마지막 2개 층만 학습(약 14% 파라미터), 같은 음식 레시피는 모두 정답 처리, WiSE-FT(기본 가중치와 α=0.5로 섞기) | 0.721 | 0.104 |
| 3차 | 추가 학습 한 번 더 시도 | — | 기본 CLIP보다 증가 |

- **1차 문제:** 학습 손실은 줄었지만 검증 손실이 1.352 → 1.505로 늘었다. ① 같은 음식의 다른 레시피를 오답으로 취급한 점, ② 전체 층을 바꿔 과적합된 점을 원인으로 봤다.
- **2차 개선:** 위 두 원인을 고쳐 검증 손실(1.953 → 1.908)과 Top-5, 주의놓침이 1차보다 좋아졌지만, 여전히 기본 CLIP보다 주의놓침이 높았다.
- **3차:** 한 번 더 시도했지만 역시 주의놓침이 늘었다.
- **결론:** 우리 레시피 데이터로 학습하면 이미 넓게 학습된 CLIP의 일반화 능력이 오히려 줄어든 것으로 해석할 수 있고, 주의놓침이 가장 낮은 **기본 CLIP을 채택**했다. 이는 사전학습 모델이 항상 더 좋다는 뜻이 아니라, 이 데이터와 이 평가 기준에서의 결과다.

## 6. 결론
- **ConvNeXt-Base (exp004):** ImageNet 사전학습 가중치를 Food-101로 전체 파인튜닝. 검증 점수로 고른 최종 모델이며 시험 Top-1 **93.46%**. 이 노트북은 앱과 같은 가중치로 시험 평가를 다시 실행한다.
- **기본 CLIP ViT-B/32:** 처음 보는 음식을 레시피 3만 5천 개에서 검색해 보완. 세 번의 추가 학습보다 주의놓침이 낮아 추가 학습 없이 사용한다.
- **두 모델을 함께 쓰는 이유:** 아는 음식은 정확한 분류기로, 모르는 음식은 이미지–텍스트 검색으로 처리해 알레르기 재료를 놓칠 위험을 줄인다.
- **한계:** 19종 알레르기 판정 규칙과 Temperature Scaling 재보정은 이 노트북에 포함하지 않았다. Temperature Scaling 값은 exp003 기록이므로 exp004 기준으로 다시 구해야 한다.

**출처:** Food-101 (Bossard et al., 2014), timm `convnext_base.fb_in22k_ft_in1k`, OpenAI `clip-vit-base-patch32`, 팀 실험 기록.